# Auditoría y análisis exploratorio
## Histórico Emergencias Montería

Notebook por etapas para auditar calidad, comportamiento operacional, patrones temporales, reincidencias, tiempos operativos, técnicos, texto libre y plan de limpieza.

**Orden sugerido de ejecución:**
1. Importación y carga
2. Etapa 1: auditoría general
3. Etapa 2: calidad de datos
4. Etapa 3: análisis descriptivo operacional
5. Etapa 4: análisis temporal
6. Etapa 5: reincidencias y análisis espacial
7. Etapa 6: tiempos operativos
8. Etapa 7: técnicos y cuadrillas
9. Etapa 8: análisis de texto
10. Etapa 9: plan de limpieza

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata
from collections import Counter
from pathlib import Path

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 180)

BASE_DIR = Path('/home/gcasta/Documentos/Proyecto')
INPUT_FILE = BASE_DIR / 'Historico Emergencias Monteria.xlsx'

DATE_COLS = [
    'Fecha registro orden',
    'Fecha asignación',
    'Fecha_llegada',
    'Fecha_control',
    'Fecha_normalizacion',
    'Fecha_finalizacion',
    'Fecha solicitud',
]

TEXT_COLS = [
    'Dirección',
    'Persona ejecuta',
    'Persona legaliza',
    'Nombre unidad operativa',
    'Causal solicitud',
    'Observacion solicitud',
    'Observación',
    'Contacto',
]

df = pd.read_excel(INPUT_FILE, sheet_name='Hoja1')
print('Shape original:', df.shape)
display(df.head(3))

## Funciones auxiliares
Estas funciones permiten reutilizar limpieza, extracción de barrio, variables temporales, métricas operativas y texto limpio en todas las etapas.

In [ ]:
def normalize_text(value):
    if pd.isna(value):
        return pd.NA
    text = str(value)
    text = text.replace('_x000D_', ' ')
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('ascii')
    text = re.sub(r'\s+', ' ', text)
    return text.strip().upper()

def extract_barrio(text):
    if pd.isna(text):
        return pd.NA
    value = str(text)
    patterns = [
        r'BARRIO:\s*([^_\n\r]+)',
        r'BARRIO\s*[:\-]\s*([^_\n\r]+)',
        r'\bBARRIO\b\s*([^_\n\r]+)',
    ]
    for pattern in patterns:
        match = re.search(pattern, value, flags=re.IGNORECASE)
        if match:
            return normalize_text(match.group(1)).strip(' .;,-')
    return pd.NA

def add_temporal_features(frame):
    registro = frame['Fecha registro orden']
    frame['anio'] = registro.dt.year
    frame['trimestre'] = registro.dt.to_period('Q').astype(str)
    frame['mes'] = registro.dt.month
    frame['semana'] = registro.dt.isocalendar().week.astype('Int64')
    frame['dia_semana'] = registro.dt.day_name()
    frame['hora'] = registro.dt.hour
    return frame

def add_operational_times(frame):
    frame['tiempo_respuesta_h'] = (frame['Fecha_llegada'] - frame['Fecha asignación']).dt.total_seconds() / 3600
    frame['tiempo_control_h'] = (frame['Fecha_control'] - frame['Fecha_llegada']).dt.total_seconds() / 3600
    frame['duracion_total_h'] = (frame['Fecha_finalizacion'] - frame['Fecha registro orden']).dt.total_seconds() / 3600
    return frame

def add_quality_flags(frame):
    frame['flag_resp_negativa'] = frame['tiempo_respuesta_h'] < 0
    frame['flag_control_negativo'] = frame['tiempo_control_h'] < 0
    frame['flag_total_negativa'] = frame['duracion_total_h'] < 0
    frame['flag_inconsistencia_temporal'] = (
        (frame['Fecha registro orden'] > frame['Fecha asignación'])
        | (frame['Fecha asignación'] > frame['Fecha_llegada'])
        | (frame['Fecha_llegada'] > frame['Fecha_control'])
        | (frame['Fecha_control'] > frame['Fecha_finalizacion'])
    )
    return frame

def build_text_features(frame):
    combined = (
        frame['Observacion solicitud'].fillna('').astype(str)
        + ' '
        + frame['Observación'].fillna('').astype(str)
    )
    combined = combined.str.replace('_x000D_', ' ', regex=False)
    combined = combined.str.replace(r'\s+', ' ', regex=True).str.strip()
    frame['texto_completo_limpio'] = combined.str.upper()
    frame['barrio_extraido'] = frame['Observacion solicitud'].map(extract_barrio)

    families = {
        'flag_excavaciones': ['EXCAV', 'ZANJA', 'RETRO', 'PERFOR', 'OBRA', 'ANDAMIO'],
        'flag_robo': ['ROBO', 'HURTO', 'HURT', 'SUSTRA', 'LADRON'],
        'flag_dano_tuberia': ['TUBER', 'ROT', 'FISUR', 'FUGA', 'VALVUL', 'ACOMETIDA'],
        'flag_obras_civiles': ['OBRA', 'CONSTRUCC', 'PAVIMENT', 'CEMENTO', 'PISO'],
    }
    for flag_name, keywords in families.items():
        pattern = '|'.join(keywords)
        frame[flag_name] = frame['texto_completo_limpio'].str.contains(pattern, case=False, regex=True, na=False)
    return frame

## Etapa 1. Auditoría general
Objetivo: comprender la estructura general del histórico antes de cualquier limpieza.

In [ ]:
for col in DATE_COLS:
    df[col] = pd.to_datetime(df[col], errors='coerce')

print('Total registros:', len(df))
print('Total columnas:', df.shape[1])
print('Memoria MB:', round(df.memory_usage(deep=True).sum() / 1024**2, 2))
print('Rango temporal:', df['Fecha registro orden'].min(), '->', df['Fecha registro orden'].max())

reg = df['Fecha registro orden']
print('\nRegistros por año')
display(reg.dt.year.value_counts().sort_index().rename_axis('Año').to_frame('Registros'))
print('\nRegistros por mes')
display(reg.dt.to_period('M').value_counts().sort_index().rename_axis('Mes').to_frame('Registros'))
print('\nRegistros por día de semana')
weekday_map = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
display(reg.dt.dayofweek.map(lambda x: weekday_map[x]).value_counts().reindex(weekday_map).rename_axis('Día').to_frame('Registros'))

In [ ]:
audit_summary = pd.DataFrame({
    'Variable': df.columns,
    'Tipo': [str(df[c].dtype) for c in df.columns],
    'Unicos': [df[c].nunique(dropna=True) for c in df.columns],
    '% Nulos': [round(df[c].isna().mean() * 100, 2) for c in df.columns],
})
display(audit_summary)

## Etapa 2. Calidad de datos
Objetivo: identificar problemas estructurales, duplicados e inconsistencias cronológicas.

In [ ]:
quality = pd.DataFrame({
    'Variable': df.columns,
    'Nulos': df.isna().sum().values,
})
quality['Unicos'] = [df[c].nunique(dropna=True) for c in df.columns]
quality['% Nulos'] = (quality['Nulos'] / len(df) * 100).round(2)
display(quality.sort_values('% Nulos', ascending=False).head(15))

exact_dups = int(df.duplicated().sum())
print('Duplicados exactos:', exact_dups)
print('Duplicados por Orden:', int(df.duplicated('Orden').sum()))
print('Duplicados por Contrato:', int(df['Contrato'].duplicated().sum()))
print('Duplicados por Direccion:', int(df['Dirección'].duplicated().sum()))

In [ ]:
viol = pd.DataFrame(index=df.index)
viol['registro_gt_asig'] = df['Fecha registro orden'] > df['Fecha asignación']
viol['asig_gt_llegada'] = df['Fecha asignación'] > df['Fecha_llegada']
viol['llegada_gt_control'] = df['Fecha_llegada'] > df['Fecha_control']
viol['control_gt_final'] = df['Fecha_control'] > df['Fecha_finalizacion']
viol['registro_gt_final'] = df['Fecha registro orden'] > df['Fecha_finalizacion']
viol_any = viol.any(axis=1)

print('Inconsistencias cronologicas totales:', int(viol_any.sum()))
display(viol.sum().rename('Casos').to_frame())
display(df.loc[viol_any, ['Fecha registro orden', 'Fecha asignación', 'Fecha_llegada', 'Fecha_control', 'Fecha_finalizacion', 'Orden', 'Contrato', 'Dirección']].head(10))

## Etapa 3. Análisis descriptivo operacional
Objetivo: cuantificar volumen y distribución de las emergencias y de sus clasificaciones principales.

In [ ]:
reg = df['Fecha registro orden'].dropna()
print('Total emergencias:', len(df))
print('Promedio mensual:', round(len(df) / reg.dt.to_period('M').nunique(), 2))
print('Promedio semanal:', round(len(df) / reg.dt.to_period('W').nunique(), 2))
print('Promedio diario:', round(len(df) / reg.dt.date.nunique(), 2))

for col in ['Tipo de Emergencia', 'Causal solicitud', 'Desc Tipo Trabajo', 'Desc estado de la orden']:
    print(f'\nDistribución por {col}')
    display(df[col].fillna('<<NULO>>').value_counts().rename_axis(col).to_frame('Casos'))

## Etapa 4. Análisis temporal
Objetivo: identificar tendencia, estacionalidad, picos y comportamiento horario.

In [ ]:
reg = df['Fecha registro orden'].dropna()
by_year = reg.dt.year.value_counts().sort_index()
by_quarter = reg.dt.to_period('Q').value_counts().sort_index()
by_month = reg.dt.to_period('M').value_counts().sort_index()
by_week = reg.dt.to_period('W').value_counts().sort_index()
by_hour = reg.dt.hour.value_counts().sort_index()

display(by_year.rename_axis('Año').to_frame('Casos'))
display(by_quarter.rename_axis('Trimestre').to_frame('Casos'))
display(by_month.rename_axis('Mes').to_frame('Casos').tail(24))
display(by_week.rename_axis('Semana').to_frame('Casos').tail(12))
display(by_hour.rename_axis('Hora').to_frame('Casos'))
print('Hora pico:', int(by_hour.idxmax()), 'con', int(by_hour.max()), 'casos')

## Etapa 5. Reincidencias y análisis espacial
Objetivo: detectar ubicaciones y contratos con comportamiento recurrente.

In [ ]:
df = build_text_features(df)

for key in ['Dirección', 'Contrato']:
    counts = df[key].fillna('<<NULO>>').value_counts()
    print(f'\nTop 20 {key}')
    display(counts.head(20).rename_axis(key).to_frame('Casos'))

display(df['barrio_extraido'].fillna('<<NO EXTRAIDO>>').value_counts().head(20).rename_axis('Barrio').to_frame('Casos'))

In [ ]:
for key in ['Dirección', 'Contrato']:
    tmp = df[[key, 'Fecha registro orden']].dropna().sort_values([key, 'Fecha registro orden'])
    gaps = tmp.groupby(key)['Fecha registro orden'].diff().dt.total_seconds() / 3600
    stats = pd.DataFrame({
        'count': tmp.groupby(key).size(),
        'gap_h_mean': gaps.groupby(tmp[key]).mean(),
        'gap_h_median': gaps.groupby(tmp[key]).median(),
    })
    display(stats[stats['count'] > 1].sort_values(['count', 'gap_h_mean'], ascending=[False, True]).head(10))

## Etapa 6. Análisis de tiempos operativos
Objetivo: evaluar desempeño con tiempos de respuesta, control y duración total.

In [ ]:
df = add_operational_times(df)
df = add_quality_flags(df)

for name, series in [('tiempo_respuesta_h', df['tiempo_respuesta_h']), ('tiempo_control_h', df['tiempo_control_h']), ('duracion_total_h', df['duracion_total_h'])]:
    s = series.dropna()
    print(f'\n{name}')
    print('mean:', round(s.mean(), 2), 'median:', round(s.median(), 2), 'std:', round(s.std(), 2), 'p90:', round(s.quantile(0.9), 2), 'p95:', round(s.quantile(0.95), 2), 'min:', round(s.min(), 2), 'max:', round(s.max(), 2))

print('Tiempos negativos por métrica')
print('respuesta_neg:', int(df['flag_resp_negativa'].sum()))
print('control_neg:', int(df['flag_control_negativo'].sum()))
print('total_neg:', int(df['flag_total_negativa'].sum()))
print('Filas con alguna inconsistencia temporal:', int(df['flag_inconsistencia_temporal'].sum()))

outliers = df[df['duracion_total_h'] > df['duracion_total_h'].quantile(0.75) + 1.5 * (df['duracion_total_h'].quantile(0.75) - df['duracion_total_h'].quantile(0.25))]
display(outliers[['Fecha registro orden', 'Fecha_finalizacion', 'Orden', 'Contrato', 'Dirección', 'duracion_total_h']].sort_values('duracion_total_h', ascending=False).head(10))

## Etapa 7. Técnicos y cuadrillas
Objetivo: evaluar carga operativa y concentración de trabajo.

In [ ]:
tech_counts = df['Persona ejecuta'].fillna('<<NULO>>').astype(str).str.strip().value_counts()
crew_counts = df['Nombre unidad operativa'].fillna('<<NULO>>').astype(str).str.strip().value_counts()
unit_counts = df['Codigo unidad de  operativa'].fillna('<<NULO>>').astype(str).str.strip().value_counts()

display(tech_counts.head(15).rename_axis('Persona ejecuta').to_frame('Ordenes'))
display(crew_counts.head(15).rename_axis('Nombre unidad operativa').to_frame('Ordenes'))
display(unit_counts.head(15).rename_axis('Codigo unidad de operativa').to_frame('Ordenes'))

In [ ]:
tech_total = df['duracion_total_h'].groupby(df['Persona ejecuta']).mean().sort_values(ascending=False)
crew_total = df['duracion_total_h'].groupby(df['Nombre unidad operativa']).mean().sort_values(ascending=False)
unit_total = df['duracion_total_h'].groupby(df['Codigo unidad de  operativa']).mean().sort_values(ascending=False)

display(tech_total.head(15).rename_axis('Persona ejecuta').to_frame('Duracion_promedio_h'))
display(crew_total.head(15).rename_axis('Nombre unidad operativa').to_frame('Duracion_promedio_h'))
display(unit_total.head(15).rename_axis('Codigo unidad de operativa').to_frame('Duracion_promedio_h'))

## Etapa 8. Análisis de texto
Objetivo: extraer información útil de las observaciones libres.

In [ ]:
texts = (df['Observacion solicitud'].fillna('') + ' ' + df['Observación'].fillna('')).astype(str)

stop_words = {
    'a','acabo','ademas','al','algo','algunas','algunos','ante','antes','aquel','aquella','aquellas','aquellos','aqui','asi','aun','bajo','bien','cabe','cada','casi','como','con','contra','cual','cuando','de','del','desde','donde','dos','el','ella','ellas','ellos','en','entre','era','erais','eran','eres','es','esa','esas','ese','eso','esos','esta','estaba','estaban','estado','estais','estamos','estan','estar','este','esto','estos','fui','fue','fueron','ha','hace','hacen','hacer','hay','incluso','la','las','le','les','lo','los','mas','me','mi','mis','mucho','muy','ni','no','nos','o','otra','otro','para','pero','poco','por','porque','pues','que','quien','se','si','sin','sobre','su','sus','te','tu','tus','un','una','uno','unos','unas','y','ya','puede','realiza','realizar','indica','informa','cliente','usuario','nombre','contrato','telefono','direccion','localidad','barrio','observaciones','observacion','tiempo','atencion','reporta','realizo','servicio','gas','tuberia'
}

def norm(text):
    text = text.lower()
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('ascii')
    text = re.sub(r'[^a-z0-9]+', ' ', text)
    return text

def tokens(text):
    return [t for t in norm(text).split() if len(t) > 2 and not t.isdigit() and t not in stop_words]

token_lists = [tokens(t) for t in texts]
counter = Counter(t for lst in token_lists for t in lst)
display(pd.Series(dict(counter.most_common(30))).rename_axis('Palabra').to_frame('Frecuencia'))

bi = Counter()
tri = Counter()
for tok in token_lists:
    bi.update(zip(tok, tok[1:]))
    tri.update(zip(tok, tok[1:], tok[2:]))
display(pd.Series({' '.join(k): v for k, v in bi.most_common(20)}).rename_axis('Bigrama').to_frame('Frecuencia'))
display(pd.Series({' '.join(k): v for k, v in tri.most_common(20)}).rename_axis('Trigrama').to_frame('Frecuencia'))

families = {
    'excavaciones': ['excav', 'zanja', 'maquina', 'retro', 'perfor', 'obra', 'andamio'],
    'robo': ['robo', 'hurt', 'ladron', 'hurto', 'sustra', 'violacion'],
    'danos_tuberia': ['tuber', 'rot', 'fractur', 'fisur', 'fuga', 'valvula', 'manguera', 'acometida'],
    'obras_civiles': ['obra', 'construcc', 'civile', 'cemento', 'paviment', 'piso', 'excav'],
}
for fam, keys in families.items():
    mask = texts.str.contains('|'.join(keys), case=False, regex=True, na=False)
    print(f'\n{fam}:', int(mask.sum()), 'registros')
    display(df.loc[mask, ['Causal solicitud', 'Observacion solicitud']].head(5))

## Etapa 9. Plan de limpieza
Objetivo: dejar definidas las reglas de limpieza antes de ejecutar una versión final depurada.

In [ ]:
cleaning_plan = pd.DataFrame([
    ['Conservar', 'Fecha registro orden', 'Variable base temporal'],
    ['Conservar', 'Fecha asignación', 'Hito operacional'],
    ['Conservar', 'Fecha_llegada', 'Hito operacional'],
    ['Conservar', 'Fecha_control', 'Hito operacional'],
    ['Conservar', 'Fecha_normalizacion', 'Hito operacional'],
    ['Conservar', 'Fecha_finalizacion', 'Cierre operacional'],
    ['Conservar', 'Tipo de Emergencia', 'Clasificación base'],
    ['Conservar', 'Orden', 'Llave operacional'],
    ['Conservar', 'Contrato', 'Llave de usuario; tratar como categórica'],
    ['Conservar', 'Dirección', 'Llave espacial a normalizar'],
    ['Conservar', 'Causal solicitud', 'Variable clave para segmentación'],
    ['Conservar', 'Observacion solicitud', 'Texto libre para NLP'],
    ['Conservar', 'Observación', 'Texto libre para NLP'],
    ['Transformar', 'Contrato', 'Identificador categórico, no numérico continuo'],
    ['Transformar', 'Orden', 'Llave operacional'],
    ['Transformar', 'Nro_solicitud', 'Identificador categórico'],
    ['Transformar', 'Interaccion', 'Código categórico'],
    ['Transformar', 'Codigo unidad de  operativa', 'Código categórico'],
    ['Transformar', 'Dirección', 'Normalización textual y geográfica'],
    ['Transformar', 'Persona ejecuta', 'Homologación de nombres'],
    ['Transformar', 'Nombre unidad operativa', 'Homologación de nombres'],
    ['Derivar', 'anio / trimestre / mes / semana / dia_semana / hora', 'Patrones temporales'],
    ['Derivar', 'tiempo_respuesta_h / tiempo_control_h / duracion_total_h', 'SLA operativo'],
    ['Derivar', 'barrio_extraido', 'Zona operativa'],
    ['Derivar', 'banderas NLP', 'Excavaciones, robo, daños de tubería, obras civiles'],
    ['Imputar', 'Persona legaliza', 'Categoría desconocido si se requiere trazabilidad'],
    ['Imputar', 'Contrato', 'Mantener nulo o categorizar como ausente'],
    ['Eliminar solo tras validación', 'Filas con inconsistencia cronológica', 'No borrar sin revisión manual'],
    ['Eliminar solo tras validación', 'Filas con fechas corruptas en 1970', 'Alta probabilidad de error de captura'],
], columns=['Acción', 'Elemento', 'Justificación'])
display(cleaning_plan)

print('Recomendación final: no eliminar todavía; auditar, marcar y luego construir matriz analítica limpia.')